<a href="https://colab.research.google.com/github/Keistkmiya/Tugas2-MachineLearning/blob/main/Tugas2_Chapter5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5: Dimensionality Reduction

## Setup and Data Loading

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

url_ev = "https://data.wa.gov/api/views/f6w7-q2d2/rows.csv?accessType=DOWNLOAD"
df_ev = pd.read_csv(url_ev)
df_ev.columns = [col.lower().replace(' ', '_') for col in df_ev.columns]
target_cols = ['model_year', 'electric_range', 'base_msrp', 'legislative_district']
numeric_columns = [col for col in target_cols if col in df_ev.columns]
print(f"Kolom numerik yang ditemukan dan akan kita gunakan: {numeric_columns}")
df_pca = df_ev.dropna(subset=numeric_columns).copy()
X = df_pca[numeric_columns].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nSetup Chapter 5 Berhasil!")
print(f"Bentuk data sekarang: {X_scaled.shape}")

Kolom numerik yang ditemukan dan akan kita gunakan: ['model_year', 'electric_range', 'legislative_district']

Setup Chapter 5 Berhasil!
Bentuk data sekarang: (280118, 3)


## Reducing Features Using Principal Components (PCA)
Principal Component Analysis (PCA) adalah teknik reduksi dimensi linier yang populer. Cara kerjanya adalah dengan memproyeksikan data kita ke arah yang memiliki variansi terbesar. Kita menentukan jumlah komponen utama yang diinginkan (misalnya kita ringkas dari 4 kolom menjadi 2 kolom). Komponen utama pertama ($PC1$) akan menangkap variansi paling banyak, diikuti oleh komponen kedua ($PC2$), dan seterusnya. Teknik ini sangat membantu kita untuk mempercepat waktu komputasi tanpa banyak mengorbankan kualitas data.

In [5]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Bentuk data asli: {X_scaled.shape}")
print(f"Bentuk data setelah PCA: {X_pca.shape}")

variansi_terjelaskan = pca.explained_variance_ratio_
print(f"\nVariansi yang ditangkap oleh PC1: {variansi_terjelaskan[0]:.2%}")
print(f"Variansi yang ditangkap oleh PC2: {variansi_terjelaskan[1]:.2%}")
print(f"Total informasi yang dipertahankan: {sum(variansi_terjelaskan):.2%}")
print("\n5 Baris pertama hasil PCA:")
print(X_pca[:5])

Bentuk data asli: (280118, 3)
Bentuk data setelah PCA: (280118, 2)

Variansi yang ditangkap oleh PC1: 51.56%
Variansi yang ditangkap oleh PC2: 33.33%
Total informasi yang dipertahankan: 84.89%

5 Baris pertama hasil PCA:
[[ 2.41833188 -1.89437275]
 [ 2.92870486 -0.39759913]
 [ 2.55230553  0.53555036]
 [-0.79956372 -1.79304763]
 [ 2.785422    1.21232273]]


## Selecting the Number of Principal Components

Menentukan jumlah komponen utama ($n\_components$) secara manual sering kali bersifat spekulatif. Teknik yang lebih profesional adalah dengan menentukan jumlah variansi (informasi) yang ingin kita pertahankan, misalnya 95% atau 99%.

Dalam Scikit-Learn, kita bisa memasukkan nilai *float* antara 0.0 hingga 1.0 pada parameter `n_components`. Algoritma akan secara otomatis memilih jumlah komponen minimum yang diperlukan untuk mencapai ambang batas variansi tersebut. Ini adalah praktik standar untuk memastikan kita tidak membuang terlalu banyak informasi penting saat melakukan kompresi data.

In [6]:
pca_95 = PCA(n_components=0.95, svd_solver='full')

X_pca_95 = pca_95.fit_transform(X_scaled)

print(f"Jumlah kolom awal: {X_scaled.shape[1]}")
print(f"Jumlah komponen yang dipilih untuk mempertahankan 95% variansi: {pca_95.n_components_}")

cumulative_variance = np.cumsum(pca_95.explained_variance_ratio_)

print("\nVariansi Kumulatif per Komponen:")
for i, v in enumerate(cumulative_variance):
    print(f"Komponen {i+1}: {v:.2%}")

Jumlah kolom awal: 3
Jumlah komponen yang dipilih untuk mempertahankan 95% variansi: 3

Variansi Kumulatif per Komponen:
Komponen 1: 51.56%
Komponen 2: 84.89%
Komponen 3: 100.00%


## Reducing Features Using Kernels (Kernel PCA)
PCA standar adalah teknik linier, yang berarti ia tidak bekerja maksimal jika struktur data kita berbentuk melingkar atau spiral. **Kernel PCA** menggunakan fungsi kernel untuk memetakan data ke ruang fitur yang berdimensi lebih tinggi di mana data tersebut menjadi linier.

Beberapa jenis kernel yang sering digunakan antara lain:
*   **RBF (Radial Basis Function):** Paling umum digunakan untuk pola melingkar. Rumusnya: $$k(x, y) = \exp(-\gamma ||x - y||^2)$$
*   **Polynomial:** Bagus untuk pola hubungan pangkat.
*   **Sigmoid:** Mirip dengan fungsi aktivasi pada neural networks.



Meskipun Kernel PCA sangat kuat, teknik ini membutuhkan daya komputasi yang lebih besar dan kita harus menentukan parameter seperti `gamma` secara manual atau melalui eksperimen.

In [7]:
from sklearn.decomposition import KernelPCA

kpca = KernelPCA(kernel="rbf", gamma=15, n_components=1)
X_kpca = kpca.fit_transform(X_scaled[:1000])

print(f"Jumlah fitur awal: {X_scaled.shape[1]}")
print(f"Jumlah fitur setelah Kernel PCA: {X_kpca.shape[1]}")
print("\n5 Baris pertama hasil Kernel PCA:")
print(X_kpca[:5])

Jumlah fitur awal: 3
Jumlah fitur setelah Kernel PCA: 1

5 Baris pertama hasil Kernel PCA:
[[-0.06807491]
 [-0.0694555 ]
 [-0.07483781]
 [-0.08340396]
 [-0.07217241]]
